# Activity 4.1 – GTSRB Traffic Sign Detection (PyTorch)

**Framework:** PyTorch + torchvision  
**Dataset:** GTSRB – German Traffic Sign Recognition Benchmark (43 classes, ~50k images)  
**Goal:** >90% classification accuracy on the validation set  
**Models compared:** Custom CNN · AlexNet · EfficientNet-B0 · ResNet-34  

> ⚠️ **Enable GPU before running:** Runtime → Change runtime type → T4 GPU

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 1 – Install dependencies
# torchvision includes a built-in GTSRB loader (no manual Kaggle download needed)
# ─────────────────────────────────────────────────────────────
!pip install -q torch torchvision tqdm scikit-learn matplotlib

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 2 – Imports
# ─────────────────────────────────────────────────────────────
import os, time, copy
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split

import torchvision
import torchvision.transforms as transforms
import torchvision.models as tv_models
from torchvision.datasets import GTSRB

from sklearn.metrics import classification_report

# Detect GPU – always prefer CUDA when available
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 3 – Hyperparameter configuration
# Edit these values to experiment with different settings
# ─────────────────────────────────────────────────────────────
IMG_SIZE    = 32          # Resize all images to 32×32 px (use 64 for higher accuracy)
NUM_CLASSES = 43          # GTSRB has exactly 43 traffic sign categories
BATCH_SIZE  = 64          # Samples per gradient update (try 32, 64, 128)
EPOCHS      = 25          # Maximum training iterations
LR          = 1e-3        # Adam learning rate (try 1e-3, 3e-4, 1e-4)
DROPOUT     = 0.4         # Dropout probability in FC layers (try 0.25, 0.4, 0.5)
PATIENCE    = 5           # Early stopping: stop after N epochs without improvement
DATA_ROOT   = './data'    # Where torchvision will download/cache the dataset

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 4 – Data transforms and loading
# torchvision.datasets.GTSRB downloads automatically on first run
# ─────────────────────────────────────────────────────────────

# Training transform: resize + data augmentation to improve generalization
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),         # Standardize all image sizes
    transforms.RandomRotation(15),                   # Random rotation ±15° (signs can be tilted)
    transforms.ColorJitter(brightness=0.3,           # Vary brightness (day/night lighting)
                           contrast=0.2,             # Vary contrast
                           saturation=0.2),          # Vary color saturation
    transforms.RandomAffine(degrees=0,               # Small horizontal/vertical shifts
                            translate=(0.1, 0.1)),
    transforms.ToTensor(),                           # Convert PIL image to float32 tensor [0,1]
    transforms.Normalize(mean=[0.3337, 0.3064, 0.3171],  # GTSRB channel means
                         std=[0.2672, 0.2564, 0.2629]),   # GTSRB channel stds
])

# Validation/test transform: only resize + normalize (no augmentation)
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.3337, 0.3064, 0.3171],
                         std=[0.2672, 0.2564, 0.2629]),
])

# Download GTSRB automatically (no Kaggle account needed)
full_train = GTSRB(root=DATA_ROOT, split='train', download=True, transform=train_transform)
test_set   = GTSRB(root=DATA_ROOT, split='test',  download=True, transform=val_transform)

# Split training into 80% train / 20% validation
val_size   = int(0.2 * len(full_train))
train_size = len(full_train) - val_size
train_set, val_set = random_split(full_train, [train_size, val_size],
                                  generator=torch.Generator().manual_seed(42))

# Apply validation transform to val_set (override augmentation)
val_set.dataset.transform = val_transform

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {train_size} | Val: {val_size} | Test: {len(test_set)}')

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 5 – Visualize a sample of training images
# ─────────────────────────────────────────────────────────────
sign_names = [
    'Speed 20','Speed 30','Speed 50','Speed 60','Speed 70',
    'Speed 80','End Speed 80','Speed 100','Speed 120','No passing',
    'No passing >3.5t','Right-of-way','Priority road','Yield','Stop',
    'No vehicles','No vehicles >3.5t','No entry','General caution','Dangerous curve left',
    'Dangerous curve right','Double curve','Bumpy road','Slippery road','Road narrows right',
    'Road work','Traffic signals','Pedestrians','Children crossing','Bicycles crossing',
    'Ice/snow','Wild animals','End restrictions','Turn right ahead','Turn left ahead',
    'Go straight','Go straight/right','Go straight/left','Keep right','Keep left',
    'Roundabout','End no passing','End no passing >3.5t'
]

imgs, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 8, figsize=(16, 5))
for i, ax in enumerate(axes.flat):
    img = imgs[i].permute(1, 2, 0).numpy()
    img = (img * [0.2672,0.2564,0.2629] + [0.3337,0.3064,0.3171]).clip(0, 1)  # unnormalize
    ax.imshow(img)
    ax.set_title(sign_names[labels[i]], fontsize=7)
    ax.axis('off')
plt.suptitle('GTSRB Sample Images', fontsize=12)
plt.tight_layout()
plt.savefig('gtsrb_samples.png', dpi=120)
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 6 – Custom CNN Architecture
# 3 convolutional blocks + BatchNorm + Dropout + FC head
# ─────────────────────────────────────────────────────────────
class CustomCNN(nn.Module):
    def __init__(self, num_classes=43, dropout=0.4):
        super().__init__()
        
        # Conv Block 1: detect low-level features (edges, color gradients)
        # Input: 3×32×32  →  Output: 32×16×16 after pooling
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),  # 32 filters, 3×3 kernel
            nn.BatchNorm2d(32),    # Normalize feature maps → stable, faster training
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),    # Downsample spatial dims by 2: 32×32 → 16×16
            nn.Dropout2d(0.25),    # Dropout2d drops entire channels (better for conv layers)
        )
        
        # Conv Block 2: detect mid-level patterns (curves, sign borders)
        # Input: 32×16×16  →  Output: 64×8×8 after pooling
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),  # Double filters to 64
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),    # 16×16 → 8×8
            nn.Dropout2d(0.25),
        )
        
        # Conv Block 3: detect high-level semantic features (sign shapes)
        # Input: 64×8×8  →  Output: 128×4×4 after pooling
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),  # 128 filters
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),    # 8×8 → 4×4
            nn.Dropout2d(0.25),
        )
        
        # Fully Connected Classification Head
        # Flatten 128×4×4 = 2048 features → 256 → 43 classes
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),  # Dense layer with 256 neurons
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),           # Stronger dropout (configurable) in FC layer
            nn.Linear(256, num_classes),   # Output: 43 class scores (logits)
        )
    
    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.classifier(x)
        return x  # Return raw logits; CrossEntropyLoss applies softmax internally

# Quick architecture check
test_model = CustomCNN(num_classes=NUM_CLASSES, dropout=DROPOUT)
test_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
print('CustomCNN output shape:', test_model(test_input).shape)  # Should be [1, 43]
total_params = sum(p.numel() for p in test_model.parameters() if p.requires_grad)
print(f'Total trainable parameters: {total_params:,}')

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 7 – Pretrained model factory
# AlexNet, ResNet-34, EfficientNet-B0 via transfer learning
# Transfer learning: use ImageNet weights, replace only the last layer
# ─────────────────────────────────────────────────────────────
def get_pretrained_model(name, num_classes=43):
    """
    Returns a pretrained model adapted for GTSRB (43 classes).
    Only the final classification layer is replaced and re-trained;
    backbone layers keep their ImageNet weights (transfer learning).
    """
    if name == 'alexnet':
        # AlexNet (Krizhevsky 2012): 5 conv layers + 3 FC layers
        m = tv_models.alexnet(weights=tv_models.AlexNet_Weights.IMAGENET1K_V1)
        m.classifier[6] = nn.Linear(4096, num_classes)  # Replace final FC: 4096 → 43
        
    elif name == 'resnet34':
        # ResNet-34: deep residual network with skip connections
        # Skip connections allow gradients to flow back through many layers
        m = tv_models.resnet34(weights=tv_models.ResNet34_Weights.IMAGENET1K_V1)
        m.fc = nn.Linear(512, num_classes)  # Replace final FC: 512 → 43
        
    elif name == 'efficientnet_b0':
        # EfficientNet-B0: compound scaling of depth/width/resolution
        # Best accuracy-to-parameter ratio among tested models
        m = tv_models.efficientnet_b0(weights=tv_models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        m.classifier[1] = nn.Linear(1280, num_classes)  # Replace final FC: 1280 → 43
    else:
        raise ValueError(f'Unknown model: {name}')
    return m

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 8 – Generic training function (reused for all models)
# ─────────────────────────────────────────────────────────────
def train_model(model, train_loader, val_loader, epochs=EPOCHS,
                lr=LR, patience=PATIENCE, model_name='model'):
    """
    Trains model, logs accuracy/loss per epoch,
    applies early stopping, and returns best weights.
    """
    model = model.to(DEVICE)
    
    # CrossEntropyLoss: combines log_softmax + NLLLoss for multi-class classification
    criterion = nn.CrossEntropyLoss()
    
    # Adam optimizer: adaptive learning rate, generally outperforms plain SGD for CNNs
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    
    # CosineAnnealingLR: gradually reduces LR following a cosine curve
    # Helps the model settle into a better local minimum in later epochs
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    
    history = {'train_acc': [], 'val_acc': [], 'train_loss': [], 'val_loss': []}
    best_val_acc = 0.0
    best_weights = None
    no_improve   = 0
    
    for epoch in range(epochs):
        # ── Training phase ──
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        
        for imgs, labels in tqdm(train_loader, desc=f'[{model_name}] Epoch {epoch+1}/{epochs}', leave=False):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()          # Clear gradients from previous step
            outputs = model(imgs)           # Forward pass
            loss = criterion(outputs, labels)  # Compute loss
            loss.backward()                # Backpropagation: compute gradients
            optimizer.step()               # Update weights
            
            running_loss += loss.item() * imgs.size(0)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total   += labels.size(0)
        
        train_loss = running_loss / total
        train_acc  = correct / total
        
        # ── Validation phase ──
        model.eval()
        val_loss_sum, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():  # Disable gradient computation for inference
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                outputs = model(imgs)
                loss = criterion(outputs, labels)
                val_loss_sum += loss.item() * imgs.size(0)
                _, predicted = outputs.max(1)
                val_correct += predicted.eq(labels).sum().item()
                val_total   += labels.size(0)
        
        val_loss = val_loss_sum / val_total
        val_acc  = val_correct  / val_total
        
        scheduler.step()  # Adjust learning rate after each epoch
        
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        
        print(f'  Epoch {epoch+1:3d} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | LR: {scheduler.get_last_lr()[0]:.6f}')
        
        # Early stopping: save best weights
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_weights = copy.deepcopy(model.state_dict())
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'  ⏹ Early stopping at epoch {epoch+1} (best val acc: {best_val_acc:.4f})')
                break
    
    model.load_state_dict(best_weights)  # Restore best weights
    torch.save(model.state_dict(), f'{model_name}_best.pth')
    print(f'  ✅ Saved best model: {model_name}_best.pth  (Val Acc: {best_val_acc:.4f})')
    return model, history, best_val_acc

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 9 – Manual hyperparameter experiment grid
# Compare different LR / dropout / batch size combinations
# on the Custom CNN before running full model comparison
# ─────────────────────────────────────────────────────────────
experiments = [
    {'name': 'CNN_lr1e-3_drop0.4',  'lr': 1e-3,  'dropout': 0.4},  # baseline
    {'name': 'CNN_lr3e-4_drop0.4',  'lr': 3e-4,  'dropout': 0.4},  # lower lr
    {'name': 'CNN_lr1e-3_drop0.5',  'lr': 1e-3,  'dropout': 0.5},  # higher dropout
    {'name': 'CNN_lr1e-3_drop0.25', 'lr': 1e-3,  'dropout': 0.25}, # lower dropout
]

exp_results = []
print('='*60)
print('HYPERPARAMETER SEARCH – Custom CNN')
print('='*60)
for cfg in experiments:
    print(f'\n▶ Experiment: {cfg["name"]}')
    m = CustomCNN(num_classes=NUM_CLASSES, dropout=cfg['dropout'])
    _, hist, best_acc = train_model(m, train_loader, val_loader,
                                     epochs=15,          # Use fewer epochs for search
                                     lr=cfg['lr'],
                                     patience=4,
                                     model_name=cfg['name'])
    exp_results.append({'Config': cfg['name'], 'LR': cfg['lr'],
                        'Dropout': cfg['dropout'], 'Best Val Acc': f"{best_acc*100:.2f}%"})

df_exp = pd.DataFrame(exp_results).sort_values('Best Val Acc', ascending=False)
print('\n' + '='*60)
print('HYPERPARAMETER SEARCH RESULTS')
print(df_exp.to_string(index=False))

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 10 – Full model comparison
# Train Custom CNN, AlexNet, ResNet-34, EfficientNet-B0
# and compare final validation accuracy
# ─────────────────────────────────────────────────────────────
model_results = []
all_histories = {}

models_to_compare = [
    ('custom_cnn',       CustomCNN(NUM_CLASSES, dropout=DROPOUT)),
    ('alexnet',          get_pretrained_model('alexnet', NUM_CLASSES)),
    ('resnet34',         get_pretrained_model('resnet34', NUM_CLASSES)),
    ('efficientnet_b0',  get_pretrained_model('efficientnet_b0', NUM_CLASSES)),
]

for name, model in models_to_compare:
    param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'\n{"="*60}')
    print(f'Training: {name}  ({param_count/1e6:.1f}M parameters)')
    print('='*60)
    
    trained_model, history, best_acc = train_model(
        model, train_loader, val_loader,
        epochs=EPOCHS, lr=LR, patience=PATIENCE, model_name=name
    )
    all_histories[name] = history
    model_results.append({'Model': name, 'Parameters': f'{param_count/1e6:.1f}M',
                          'Best Val Accuracy': f'{best_acc*100:.2f}%',
                          '>90% Target': '✅' if best_acc > 0.90 else '❌'})

# Print comparison table
df_models = pd.DataFrame(model_results)
print('\n' + '='*60)
print('MODEL COMPARISON RESULTS')
print(df_models.to_string(index=False))

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 11 – Plot training curves for all models
# ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
colors = ['#2196F3', '#FF5722', '#4CAF50', '#9C27B0']

for (name, hist), color in zip(all_histories.items(), colors):
    axes[0].plot(hist['val_acc'],  label=name, color=color, linewidth=2)
    axes[1].plot(hist['val_loss'], label=name, color=color, linewidth=2)

axes[0].axhline(0.90, color='red', linestyle='--', linewidth=1, label='90% target')
axes[0].set_title('Validation Accuracy per Epoch', fontsize=13)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_title('Validation Loss per Epoch', fontsize=13)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('GTSRB – Model Comparison Training Curves', fontsize=14)
plt.tight_layout()
plt.savefig('model_comparison_curves.png', dpi=150)
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 12 – Evaluate best model on official test set
# ─────────────────────────────────────────────────────────────
# Load best model (pick the one with highest val accuracy from comparison)
best_model_name = df_models.sort_values('Best Val Accuracy', ascending=False).iloc[0]['Model']
print(f'Best model: {best_model_name}')

# Reload it from saved weights
if best_model_name == 'custom_cnn':
    best_final = CustomCNN(NUM_CLASSES, dropout=DROPOUT)
else:
    best_final = get_pretrained_model(best_model_name, NUM_CLASSES)

best_final.load_state_dict(torch.load(f'{best_model_name}_best.pth', map_location=DEVICE))
best_final = best_final.to(DEVICE).eval()

# Full evaluation on official test set
all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(DEVICE)
        outputs = best_final(imgs)
        _, preds = outputs.max(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

test_acc = np.mean(np.array(all_preds) == np.array(all_labels))
print(f'\n✅ Final Test Accuracy ({best_model_name}): {test_acc*100:.2f}%')
print(f'Target (>90%): {"PASSED ✅" if test_acc > 0.90 else "FAILED ❌"}')
print('\nDetailed Classification Report (first 10 classes):')
print(classification_report(all_labels, all_preds,
                             target_names=[sign_names[i] for i in range(NUM_CLASSES)],
                             labels=list(range(10))))

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 13 – Export model for Webots controller
# ─────────────────────────────────────────────────────────────

# Option A: Save as TorchScript (recommended for Webots)
# TorchScript serializes the model without needing the class definition at load time
scripted = torch.jit.script(best_final.cpu())
scripted.save('gtsrb_model_scripted.pt')
print('Saved: gtsrb_model_scripted.pt  (use with torch.jit.load in Webots)')

# Option B: Export to ONNX (universal format, works with C++ / other frameworks)
dummy_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
torch.onnx.export(best_final.cpu(), dummy_input, 'gtsrb_model.onnx',
                  input_names=['image'], output_names=['class_scores'],
                  opset_version=11)
print('Saved: gtsrb_model.onnx         (universal ONNX format)')

# Save label mapping for the Webots controller
import json
with open('sign_names.json', 'w') as f:
    json.dump(sign_names, f, indent=2)
print('Saved: sign_names.json          (class index → sign name mapping)')

# Save experiment results as CSV for the report
df_models.to_csv('model_comparison_results.csv', index=False)
df_exp.to_csv('hyperparam_search_results.csv', index=False)
print('Saved: model_comparison_results.csv')
print('Saved: hyperparam_search_results.csv')